In [1]:
import numpy as np
import pandas as pd

class DecisionTreeScratch:
    def __init__(self, max_depth=3):
        self.max_depth = max_depth
        self.tree = None

    def fit(self, X, y, depth=0):
        if depth >= self.max_depth or len(y) <= 2:
            return np.mean(y)

        best_mse = float('inf')
        best_split = None

        for col in range(X.shape[1]):
            thresholds = np.unique(X[:, col])
            for t in thresholds:
                left_mask = X[:, col] <= t
                right_mask = ~left_mask
                if not any(left_mask) or not any(right_mask): continue
                
                mse = np.var(y[left_mask]) * sum(left_mask) + np.var(y[right_mask]) * sum(right_mask)
                if mse < best_mse:
                    best_mse, best_split = mse, (col, t)

        if best_split is None: return np.mean(y)

        col, t = best_split
        left = self.fit(X[X[:, col] <= t], y[X[:, col] <= t], depth + 1)
        right = self.fit(X[X[:, col] > t], y[X[:, col] > t], depth + 1)
        return {"col": col, "t": t, "left": left, "right": right}

    def predict_one(self, x, node):
        if not isinstance(node, dict): return node
        if x[node["col"]] <= node["t"]:
            return self.predict_one(x, node["left"])
        else:
            return self.predict_one(x, node["right"])

class RandomForestScratch:
    def __init__(self, n_trees=5, max_depth=3):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.trees = []

    def fit(self, X, y):
        self.trees = []
        for _ in range(self.n_trees):
            idx = np.random.choice(len(X), len(X), replace=True)
            tree = DecisionTreeScratch(max_depth=self.max_depth)
            tree.tree = tree.fit(X[idx], y[idx])
            self.trees.append(tree)
        print(f"Đã huấn luyện xong {self.n_trees} RandomForest!")

    def predict(self, X):
        tree_preds = np.array([[t.predict_one(x, t.tree) for t in self.trees] for x in X])
        return np.mean(tree_preds, axis=1)

np.random.seed(42)
n_samples = 150


X_data = np.column_stack([
    np.random.uniform(5, 50, n_samples),      # Giá sản phẩm
    np.random.randint(0, 4, n_samples),      # Mùa (0: Xuân, 1: Hạ, 2: Thu, 3: Đông)
    np.random.randint(1, 10, n_samples)      # Độ phổ biến (1-10)
])

y_data = (X_data[:, 0] * 1.2) + (X_data[:, 2] * 5) + np.random.normal(0, 2, n_samples)
indices = np.arange(n_samples)
np.random.shuffle(indices)
train_idx, test_idx = indices[:120], indices[120:]

X_train, X_test = X_data[train_idx], X_data[test_idx]
y_train, y_test = y_data[train_idx], y_data[test_idx]

my_rf = RandomForestScratch(n_trees=10, max_depth=5)
my_rf.fit(X_train, y_train)
predictions = my_rf.predict(X_test)

print("\n" + "="*40)
print(f"{'Thực tế':>15} | {'Dự báo':>15} | {'Chênh lệch':>15}")
print("-" * 50)
for i in range(10):
    diff = abs(y_test[i] - predictions[i])
    print(f"{y_test[i]:>15.2f} | {predictions[i]:>15.2f} | {diff:>15.2f}")
mse = np.mean((y_test - predictions)**2)
print("="*40)
print(f"Lỗi bình phương trung bình (MSE): {mse:.2f}")

Đã huấn luyện xong 10 RandomForest!

        Thực tế |          Dự báo |      Chênh lệch
--------------------------------------------------
          85.85 |           83.13 |            2.72
          46.00 |           45.22 |            0.78
          44.98 |           50.37 |            5.39
          56.87 |           56.61 |            0.26
          71.57 |           75.09 |            3.52
          81.97 |           73.19 |            8.78
          88.57 |           87.51 |            1.05
          22.46 |           24.52 |            2.06
          70.55 |           65.98 |            4.57
          13.88 |           19.61 |            5.74
Lỗi bình phương trung bình (MSE): 27.67
